# Trabalho Prático de Inteligência Artificial 2025/2026
## PopOut + MCTS + ID3

**Disciplina:** Inteligência Artificial — 2025/2026
**Data:** Maio 2026
**Grupo:** _[preencher]_

---

Este notebook documenta o trabalho prático integralmente. Está escrito para alguém que **nunca viu o projeto** poder seguir do início ao fim: o problema, as decisões, os algoritmos, a implementação, os resultados. Os blocos de código correm contra o motor real (em `codes/`) — não são pseudocódigo.

A leitura é sequencial. Cada secção fecha um capítulo do trabalho antes de abrir o seguinte.

## Sumário

1. **O problema** — o que é o PopOut e o que é pedido.
2. **Decisões de design** — o que fixámos antes de programar e porquê.
3. **Motor do jogo** — board, regras, código.
4. **Interface e estratégias** — como humanos e algoritmos jogam pelo mesmo loop.
5. **MCTS standard** — os 4 passos, UCB1, validação numérica.
6. **Variações do MCTS** — sweeps de rollout, N, C, max_children.
7. **Tactical lookahead** — o "patch" que apanha pops vencedores.
8. **ID3 + Iris** — árvore de raiz, discretização, resultados.
9. **Geração do dataset PopOut** — behavioural cloning.
10. **ID3 sobre PopOut** — sweep, tree strategy, fallback.
11. **Avaliação experimental** — win-rate matrix, learning curve, tempos.
12. **Conclusões e trabalho futuro.**
13. **Referências.**

## 1. O problema

### 1.1 PopOut em três frases

PopOut é uma variante do Connect-4 descrita por James D. Allen no livro *The Complete Book of Connect-4* (2010). O jogo decorre num tabuleiro 7 colunas × 6 linhas e os jogadores alternam entre dois tipos de jogada: **drop** (largar uma peça pelo topo, como no Connect-4 tradicional) ou **pop** (puxar uma peça da própria cor do fundo da coluna, fazendo cair tudo o que está em cima). Vence quem alinhar primeiro 4 peças seguidas (horizontal, vertical ou diagonal).

A novidade está nos pops, e nas três regras especiais que tratam casos ambíguos:

1. **Pop com duplo-4 simultâneo:** se um pop fecha 4-em-linha para os dois jogadores ao mesmo tempo, vence quem fez o pop. O 4 do adversário é ignorado.
2. **Empate por tabuleiro cheio:** se o tabuleiro está cheio e o jogador a mover não tem pops legais, é empate.
3. **Empate por repetição tripla:** se o mesmo estado (board + jogador a mover) aparece 3 vezes na mesma partida, qualquer jogador pode reclamar empate.

### 1.2 O que o enunciado pede

O trabalho tem **duas metades** que se cruzam no fim:

**Metade A — Pesquisa adversarial.** Implementar PopOut e fazer uma máquina jogar bem, usando **Monte Carlo Tree Search (MCTS) com UCB1**. Não basta a versão standard — o enunciado exige *"analyse/explore different numbers of selected children for each node, and other strategies"*.

**Metade B — Aprendizagem indutiva.** Implementar **ID3** de raiz (sem `scikit-learn`) e aplicá-lo a dois datasets: o clássico Iris (numérico, 3 classes — exige discretização) e um dataset gerado pelo próprio MCTS — pares (estado, jogada) — para treinar uma árvore que **imita** o MCTS (*behavioural cloning*).

**Três cenários de jogo obrigatórios:**
- Humano vs. Humano
- Humano vs. Computador
- Computador vs. Computador com **dois algoritmos diferentes** (no nosso caso: MCTS vs. árvore-aprendida-do-MCTS).

## 2. Decisões de design

O enunciado, no §4.7, pede explicitamente que mencionemos os *constraints* da nossa solução. Fixámo-los **antes de escrever código** — esta secção lista cada decisão e a razão.

### 2.1 Tabuleiro: 6×7 com NumPy `int8`

**Dimensões 7×6** porque é o tamanho clássico do Connect-4 (e o da figura no enunciado), permitindo validar contra a literatura existente. **Representação interna** é uma matriz NumPy `(6, 7)` de `int8` — 0 = vazio, 1 = P1, 2 = P2. Linha 0 é o topo, linha 5 é o fundo. Esta orientação mantém a intuição visual (imprime-se de cima para baixo) e simplifica o `_drop_row`.

Alternativa rejeitada: **bitboards** (board em 2 inteiros de 64 bits, um por jogador). Seriam ~10× mais rápidos para milhões de simulações mas tornam o código difícil de depurar. Para o nosso volume (`N=200` em ~1s), NumPy chega.

### 2.2 Estado imutável (`frozen dataclass`)

Cada `apply_move` devolve **um novo** `State`, nunca muta o original. Razão crítica: o MCTS partilha estados entre nós da árvore. Se um rollout mutasse o board no sítio errado, todos os nós ascendentes ficariam corrompidos silenciosamente. A imutabilidade torna esse bug **impossível por construção**.

### 2.3 Estratégia como função

Cada jogador é uma função `state -> Move | str`. O `play_game(p1, p2)` chama-a quando é a vez do jogador. **Os 3 cenários do enunciado saem todos do mesmo loop** — só mudam os argumentos. É o padrão *strategy* aplicado aos jogadores.

### 2.4 ID3 de raiz, sem `scikit-learn`

Exigência do enunciado. Bibliotecas de manipulação de dados (pandas/numpy) são permitidas; o algoritmo de aprendizagem tem de ser nosso. Implementámos os **4 casos base canónicos** de Russell & Norvig — ver §8.

### 2.5 Discretização explícita

ID3 puro só aceita atributos categóricos. Iris é numérico → discretizamos antes de treinar. Implementámos 3 estratégias para comparar (`equal_width`, `equal_frequency`, `supervised`); a escolha empírica está documentada na §8.

### 2.6 Interface CLI + GUI Pygame

CLI cumpre o requisito mínimo do enunciado (formato `X--O-XO`). A GUI é um extra que torna a defesa mais demonstrável e suporta os 7 modos (incluindo "MCTS vs Tree", que é o cenário CvC com 2 algoritmos do enunciado).

### 2.7 Reprodutibilidade

Todas as estratégias aceitam um `random.Random` como parâmetro. Em nenhum sítio usamos o `random` global. Cada experiência declara o seu seed → resultados são reproduzíveis bit-a-bit.

## 3. Motor do jogo (`codes/popout.py`)

O motor é a peça em que tudo o resto assenta. Define o estado, as jogadas legais, as transições e a deteção de fim de jogo.

### 3.1 Tipos centrais

```python
@dataclass(frozen=True)
class Move:
    column: int
    kind: str  # 'drop' or 'pop'

@dataclass(frozen=True)
class State:
    board: np.ndarray              # (6, 7) int8
    player_to_move: int             # 1 ou 2
    history_counts: Dict[bytes, int]  # para repetição tripla
    last_move: Optional[Move]
    winner: Union[int, str, None]   # None, 1, 2 ou 'draw'
```

A `history_counts` é a parte engenhosa. Indexa-se por `state_key(board, player_to_move)` (uma serialização de bytes), o que permite detectar repetição tripla em **O(1)** — sem precisar de comparar boards um a um.

### 3.2 Funções principais

**`legal_moves(state)`** — para cada coluna `c`: drop é legal se a célula do topo está vazia; pop é legal se a célula do fundo é da cor do jogador atual.

**`apply_move(state, move)`** — copia o board, aplica a jogada, atualiza `history_counts`, chama `check_win`, devolve um novo `State`. Para um pop, faz uma *shift down* vetorizada com NumPy: `new_board[1:6, c] = old[0:5, c]; new_board[0, c] = EMPTY`.

**`check_win(board, last_move, mover)`** — codifica a regra do duplo-4 de Allen 2010:

```python
if last_move.kind == "pop":
    if me_won:    return mover    # pop player wins, mesmo se ambos alinharam 4
    if other_won: return other
else:  # drop
    if me_won:    return mover    # adversário não pode "ganhar" num drop nosso
```

Repara que num drop só o jogador que largou pode completar quatro — não há como o adversário formar 4 com a tua peça. Mas num pop, o board todo desce e ambos podem completar 4 simultaneamente. Daí o caso especial.

**`can_claim_repetition_draw(state)`** — devolve `True` se `history_counts[state_key] >= 3`.

In [ ]:
# Demo: uma partida scriptada que termina em vitória horizontal de P1
from popout import initial_state, apply_move, Move, render

state = initial_state()
moves = [
    Move(0, 'drop'),  # P1 → (5,0)
    Move(6, 'drop'),  # P2 → (5,6)
    Move(1, 'drop'),  # P1 → (5,1)
    Move(6, 'drop'),  # P2 → (4,6)
    Move(2, 'drop'),  # P1 → (5,2)
    Move(6, 'drop'),  # P2 → (3,6)
    Move(3, 'drop'),  # P1 → (5,3) — fecha 4-em-linha horizontal
]
for m in moves:
    state = apply_move(state, m)

print(render(state.board))
print(f'\nVencedor: P{state.winner}')

## 4. Interface e estratégias (`codes/game.py`, `codes/gui.py`)

### 4.1 O loop universal

`play_game(p1, p2)` é a abstração central. Cada jogador é um callable `state -> Move | str`. Strings especiais: `"resign"` (desiste, dá vitória ao outro) e `"draw"` (reclama empate por repetição tripla, só válido se `can_claim_repetition_draw` devolver `True`).

```python
play_game(human_strategy, human_strategy)        # H vs H
play_game(human_strategy, mcts_strategy(...))    # H vs C
play_game(mcts_strategy(...), tree_strategy(t))  # C vs C com 2 algoritmos
```

### 4.2 CLI

`game.py` traz um menu com 7 modos (espelho dos da GUI). Comandos no prompt:

| Input | Acção |
|---|---|
| `0`–`6` | drop na coluna (atalho) |
| `d 3` | drop explícito na coluna 3 |
| `p 0` | pop na coluna 0 |
| `draw` | reclama empate por repetição tripla |
| `q` ou `quit` | resigna |
| `?` ou `help` | mostra ajuda |

Lançar com `cd codes && python game.py`.

### 4.3 GUI Pygame

`gui.py` traz cliques nos botões drop/pop, modo menu, indicador "AI thinking" e *score* persistente entre partidas. O MCTS corre num thread daemon (`StrategyWorker`) para a UI não congelar. Lançar com `python gui.py`.

## 5. MCTS standard (`codes/mcts.py`)

### 5.1 Porquê MCTS e não MiniMax/Alpha-Beta?

PopOut tem branching factor até **14** (7 drops + 7 pops) e **não temos uma função de avaliação heurística estabelecida** — o Connect-4 standard tem várias publicadas, mas a dinâmica do pop estraga essas heurísticas (uma posição "boa" antes do pop pode virar terrível depois). MiniMax exige uma função de avaliação para cortar; sem isso, `b^d` explode mesmo a profundidades modestas.

MCTS é *domain-agnostic* — só precisa de simulações aleatórias até ao fim do jogo. É *anytime* (mais simulações = melhor, mas dá uma resposta a qualquer momento). E tem garantias matemáticas (UCB1, regret logarítmico).

### 5.2 Os quatro passos

```
para i = 1..N:
    1. Selection      — desce pela árvore escolhendo filho com maior UCB1
    2. Expansion      — chega a folha; cria filho novo a partir de jogada não tentada
    3. Simulation     — rollout aleatório até ao fim do jogo
    4. Backpropagation — propaga o resultado para cima (atualiza N e U em cada nó)

devolve: jogada do filho da raiz com mais visitas
```

### 5.3 Fórmula UCB1

$$\text{UCB1}(n) = \underbrace{\frac{U(n)}{N(n)}}_{\text{exploitation}} + C \cdot \underbrace{\sqrt{\frac{\ln N(\text{parent}(n))}{N(n)}}}_{\text{exploration}}$$

- $U(n)$ — vitórias acumuladas em $n$ **do ponto de vista do jogador que escolheu vir aqui** (o jogador que estava a mover no nó pai).
- $N(n)$ — número de visitas a $n$.
- $C$ — constante de exploração; canónico **$C = \sqrt{2} \approx 1.414$** (Auer, Cesa-Bianchi, Fischer 2002, derivado da desigualdade de Hoeffding).

**Por que `ln`?** Cresce devagar — à medida que o pai é visitado mais e mais vezes, o bónus de exploração para os filhos pouco visitados sobe, mas devagar. Evita que filhos promissores sejam esquecidos cedo, sem desperdiçar simulações em filhos claramente inferiores depois de muitos dados.

### 5.4 A convenção de sinal — o erro #1 a evitar

`U` é actualizado do ponto de vista do **jogador que escolheu vir** ao nó (= `parent.player_to_move`). Empates valem **0.5**. Se trocares isto, o algoritmo maximiza do lado errado em níveis ímpares.

```python
# mcts.py:backprop
def backprop(leaf, winner):
    node = leaf
    while node is not None:
        node.N += 1
        if node.parent is not None:
            chooser = node.parent.state.player_to_move
            if winner == chooser:
                node.U += 1.0
            elif winner == "draw":
                node.U += 0.5
            # else (loss): U += 0
        node = node.parent
```

### 5.5 Decisão final: filho mais visitado, não maior `U/N`

No fim das N simulações, devolvemos a jogada do filho da raiz com mais **visitas**, não com melhor **taxa**. Razão: um filho com `65/100` (65% win-rate, 100 visitas) é mais fiável do que um com `2/3` (67% win-rate, 3 visitas — variância enorme). UCB1 naturalmente concentra visitas no melhor filho à medida que `N` cresce. *Most-visited* é a estatística mais robusta (Browne et al. 2012).

### 5.6 Validação numérica do UCB1

Replicámos o exemplo trabalhado dos slides (MATERIA_IA §11.4) num teste unitário. Pai com $N=100$, três filhos:

| Filho | $U$ | $N$ | exploit | explore | UCB1 |
|---|---|---|---|---|---|
| X | 20 | 25 | 0.800 | 0.607 | **1.407** |
| Y | 30 | 70 | 0.429 | 0.363 | **0.791** |
| Z | 2  | 5  | 0.400 | 1.357 | **1.758** ← seleccionado |

Z vence apesar do exploit pior — pouco explorado, o termo de exploração domina. Se a nossa fórmula tivesse um erro de sinal, log natural-vs-base-2, ou raiz, este teste falhava.

In [ ]:
# Demo: o MCTS encontra a vitória imediata
import math, random
from popout import initial_state, apply_move, Move
from mcts import mcts_search

# P1 com 3-em-linha em (5, 0..2); vez do P1
state = initial_state()
seq = [
    Move(0, 'drop'), Move(0, 'drop'),
    Move(1, 'drop'), Move(1, 'drop'),
    Move(2, 'drop'), Move(6, 'drop'),
]
for m in seq:
    state = apply_move(state, m)

chosen = mcts_search(state, n_simulations=300, c=math.sqrt(2),
                     rng=random.Random(0))
print(f'MCTS escolheu: {chosen}')
print('Esperado: drop(3) — completa 4-em-linha em row 5')

## 6. Variações do MCTS

O enunciado pede *"analyse/explore different numbers of selected children for each node, and other strategies, not only keeping the standard implementation"*. Implementámos **4 variações** controláveis pelos parâmetros do `mcts_search`:

### 6.1 As 4 variações implementadas

| Variação | Parâmetro | Valores testados |
|---|---|---|
| Rollout policy | `rollout` | `"random"`, `"heuristic_win"`, `"heuristic_block"` |
| Constante de exploração | `c` | 0.5, 1.0, √2, 2.0 |
| Número de simulações | `n_simulations` | 100, 200, 300, 500, 600, 1000 |
| Limite de filhos | `max_children` | None, 5, 3 (com prioridade central) |

### 6.2 Rollout policies

- **`random`** — joga aleatoriamente até ao fim. Custo O(1)/ply. Baseline.
- **`heuristic_win`** — se houver vitória imediata, agarra-a; senão, random. Custo O(b)/ply.
- **`heuristic_block`** — agarra vitórias imediatas; senão, prefere jogadas que **não** dão vitória ao adversário a seguir. Custo O(b²)/ply.

### 6.3 Resultados — Experiência A (rollout policy)

10 partidas alternando lados, N=200 para todos:

| A | B | A | B | t/move A | t/move B |
|---|---|---|---|---|---|
| `random` | `heuristic_win` | 0 | **6** | 244 ms | 643 ms |
| `random` (N=300) | `heuristic_block` (N=50) | **4** | 0 | 416 ms | 6336 ms |
| `heuristic_win` (N=200) | `heuristic_block` (N=50) | **4** | 0 | 799 ms | 7266 ms |

**Conclusão:** `heuristic_win` é o sweet-spot. `heuristic_block` é demasiado caro — ao mesmo orçamento de tempo, perde para `heuristic_win` com mais simulações.

### 6.4 Resultados — Experiência B (número de simulações)

| A | B | A | B |
|---|---|---|---|
| N=100 | N=300 | 0 | **6** |
| N=300 | N=600 | 0 | **4** |

Crescimento monotónico. Sem saturação visível até N=600.

### 6.5 Resultados — Experiência C (constante C)

Vs `random`, N=200, rollout `heuristic_win`:

| C | Win-rate vs random | t/move |
|---|---|---|
| 0.5 | 6/6 | 728 ms |
| 1.0 | 6/6 | 648 ms |
| **√2** | **6/6** | **533 ms** |
| 2.0 | 6/6 | 876 ms |

Todos saturam contra random. **C=√2 é o mais rápido** — suporta empiricamente o canónico.

### 6.6 Resultados — Experiência D (max_children)

Vs `random`, N=200, prioridade central nas colunas:

| k | Win-rate vs random | t/move |
|---|---|---|
| None | 4/4 | 527 ms |
| 5 | 4/4 | 438 ms |
| **3** | **4/4** | **356 ms** |

Limitar a 3 filhos centrais mantém a qualidade e poupa **33% de tempo**.

## 7. Tactical lookahead — Fase 4.5

### 7.1 Porque foi preciso

Durante os testes do MCTS standard, observámos um problema empírico: **a IA não fazia pops mesmo quando popar era a melhor jogada**. A causa é estatística:

- Os rollouts são aleatórios. A probabilidade duma sequência específica de pops levar a vitória num rollout random é minúscula.
- Logo, jogadas tipo "pop com vitória forçada em 3 plies" recebem `U/N` baixo — o MCTS sub-estima-as e não as escolhe.

### 7.2 A solução

Antes de chamar o MCTS, faz-se uma **busca exata 2-ply** que devolve:

1. Uma jogada que vence imediatamente (drop ou pop), se existir.
2. Uma jogada que cria **fork** — todas as respostas do adversário continuam a perder para mim, se existir.
3. Caso contrário, segue para o MCTS normal.

```python
# Pseudo-código de mcts.py:find_forced_win
def find_forced_win(state, depth=2):
    if state.winner is not None: return None
    moves = legal_moves(state)
    if win_now := first move that immediately wins: return win_now
    if depth == 1: return None
    for m in moves:
        ns = apply_move(state, m)
        if all opponent replies leave me with immediate win: return m   # fork
    return None
```

### 7.3 Custo

O(b³) na raiz ≈ ~3 000 `apply_move` por jogada ≈ **20-100 ms**. Negligível ao lado do MCTS (~1 s). Activa-se com `tactical_root=True` no `mcts_search`.

### 7.4 Por que isto importa

A justificação que tens de saber: **rollouts aleatórios têm bias contra jogadas raras com payoff alto**. O tactical lookahead complementa exatamente onde o MCTS estatístico falha — em cenários determinísticos curtos.

In [ ]:
# Demo: tactical lookahead apanha um fork que o MCTS sozinho subestimaria
import math, random
from popout import initial_state, apply_move, Move
from mcts import find_forced_win, mcts_search

state = initial_state()
seq = [
    Move(2, 'drop'),  # P1 (5,2)
    Move(0, 'drop'),  # P2 (5,0)
    Move(4, 'drop'),  # P1 (5,4)
    Move(6, 'drop'),  # P2 (5,6)
]
for m in seq:
    state = apply_move(state, m)

# Sem MCTS, só com tactical lookahead:
forced = find_forced_win(state, depth=2)
print(f'Tactical lookahead encontrou: {forced}')

# Com tactical_root=True, o MCTS apanha mesmo com N=10:
chosen = mcts_search(state, n_simulations=10, c=math.sqrt(2),
                     rollout='random', tactical_root=True,
                     rng=random.Random(0))
print(f'MCTS+tactical: {chosen}')

## 8. Árvores de decisão (ID3) + Iris (`codes/decision_tree_builder.py`)

### 8.1 O algoritmo, em palavras

Tens uma tabela de exemplos rotulados (Iris: 150 flores × 4 medidas + espécie). Queres uma árvore que faça uma sequência de perguntas tipo "petallength alto?" para classificar uma flor nova. ID3 (Quinlan 1986) escolhe, em cada nó, **a pergunta que mais reduz a incerteza** (information gain) e depois recurre.

### 8.2 Os 4 casos base canónicos

```
ID3(X, y, atributos, parent_majority):
    1. Se y == ∅ → folha(parent_majority)              # subset vazio
    2. Se y é puro (1 classe) → folha(essa classe)
    3. Se atributos == ∅ ou depth >= max_depth → folha(maioria(y))
    4. A* = argmax_a IG(X, y, a)
       Se IG(A*) <= 0 → folha(maioria(y))               # nada separa
       Senão: cria nó com teste em A*; recursão por valor único de A*
```

### 8.3 Information Gain

**Entropia da classe `C`** (incerteza em bits):

$$H(C) = -\sum_c P(c) \log_2 P(c)$$

Se 50/50 entre 2 classes, $H = 1$ (incerteza máxima). Se todas iguais, $H = 0$.

**Information Gain** de um atributo $A$ é a redução de entropia ao splitar por $A$:

$$\text{Gain}(A) = H(C) - \sum_v \frac{|S_v|}{|S|} H(C \mid A=v)$$

### 8.4 Validação numérica — exemplo "Gripe"

Replicámos o exercício 11.5 dos slides:
- $H(C) = 0.954$ bits ✓
- $\text{Gain}(\text{Febre}) = 0.549$ ✓
- $\text{Gain}(\text{Tosse}) = \text{Gain}(\text{Cansaço}) = 0.159$ ✓

### 8.5 Discretização — porque é necessária

ID3 puro só aceita atributos categóricos. Iris tem 4 atributos numéricos contínuos → temos de **discretizar** antes de treinar. Implementámos 3 estratégias:

| Estratégia | Como funciona |
|---|---|
| `equal_width` | `np.linspace(min, max, k+1)` → bins iguais |
| `equal_frequency` | quantis com `np.quantile` → bins com igual nº de amostras |
| `supervised` | threshold binário que maximiza Information Gain |

### 8.6 Resultados Iris — split 80/20

| Estratégia | Acc treino | Acc teste | Folhas | Profundidade |
|---|---|---|---|---|
| supervised (binário) | 0.717 | 0.633 | **4** | 3 |
| **equal_width(k=3)** | **0.992** | **0.933** | 5 | 3 |
| equal_width(k=5) | 0.967 | 0.900 | 17 | 4 |
| equal_frequency(k=3) | 0.992 | 0.933 | 10 | 4 |

### 8.7 Cross-validation 5-fold

| Estratégia | Mean acc | Std acc | Mean folhas |
|---|---|---|---|
| supervised | 0.653 | ±0.072 | 4.2 |
| **equal_width(k=3)** | **0.947** | **±0.040** | 8.4 |
| equal_freq(k=3) | 0.947 | ±0.045 | 12.6 |

### 8.8 Por que escolhemos `equal_width(k=3)`

A discretização `supervised` produz a árvore mais pequena (4 folhas) — cumpre literalmente o "minimizar tamanho" do enunciado. **Mas comprime demais:** cada feature vira binária e perde-se informação, caindo a accuracy para 65%. `equal_width(k=3)` adiciona apenas **+4 folhas** mas ganha **+30 pp** em accuracy. É o trade-off mais favorável.

A intuição: 3 níveis (low / mid / high) correspondem aos 3 clusters reais das espécies de Iris. 2 níveis comprimem demais; 5 introduz fragmentação.

In [ ]:
# Pipeline Iris completo: ler → split → discretizar → treinar → avaliar
import pandas as pd
from decision_tree_builder import (
    fit_discretizer_equal_width, transform_discretizer,
    id3, predict_batch, accuracy, render_tree_text, tree_size,
)

df = pd.read_csv('iris.csv').drop(columns=['ID'])
feats = [c for c in df.columns if c != 'class']

df_shuf = df.sample(frac=1, random_state=0).reset_index(drop=True)
sp = int(0.8 * len(df_shuf))
X_tr, y_tr = df_shuf.iloc[:sp][feats], df_shuf.iloc[:sp]['class']
X_te, y_te = df_shuf.iloc[sp:][feats], df_shuf.iloc[sp:]['class']

fit = fit_discretizer_equal_width(X_tr, feats, n_bins=3)
tree = id3(transform_discretizer(X_tr, fit), y_tr, feats)
acc = accuracy(y_te, predict_batch(tree, transform_discretizer(X_te, fit)))
sz = tree_size(tree)

print(f'Accuracy de teste: {acc:.3f}')
print(f'Tamanho da árvore: {sz}')
print()
print('Estrutura da árvore:')
print(render_tree_text(tree))

## 9. Geração do dataset PopOut (`codes/generate_dataset.py`)

### 9.1 Behavioural cloning — o conceito

A ideia é simples: o **MCTS é o professor** (toma decisões com base em centenas de simulações) e a **árvore ID3 é o aluno** (aprende a aproximar a política do MCTS sem precisar de simulações em runtime).

**Vantagem prática:** o MCTS demora ~150-700 ms por jogada. A árvore decide em microssegundos. Para um cenário onde o tempo importa (jogo em tempo real), esta troca pode valer a pena — sacrificamos algo de qualidade pelo enorme ganho de velocidade.

**Limitação esperada:** a árvore só pode imitar o que viu no treino. Em estados raros, o MCTS investiga em runtime; a árvore "dispara" a regra mais próxima. E a árvore reflecte os erros do MCTS — não os filtra.

### 9.2 O pipeline

```
para cada partida em 1..50:
    estado ← inicial
    enquanto não terminado:
        com probabilidade ε=0.10: jogada ← random          # exploração
        senão:                    jogada ← MCTS(estado)
        gravar (estado, to_play, jogada)
        estado ← apply_move(estado, jogada)
```

A **ε-exploração** é crítica. Sem ela, MCTS-vs-MCTS com seeds determinísticos converge para a mesma partida e o dataset teria pouca cobertura. ε=0.10 garante variedade no espaço de estados.

### 9.3 Encoding

| Coluna | Tipo | Domínio |
|---|---|---|
| `s0..s41` | int | {0, 1, 2} (cells em row-major: s0=(0,0), s7=(1,0), …, s41=(5,6)) |
| `to_play` | int | {1, 2} |
| `move` | string | `d0`–`d6` ou `p0`–`p6` |

Encoding **raw** (não derivado) é uma decisão consciente: queremos que a árvore aprenda o que for relevante, sem viés humano embutido nas features. Discutimos as consequências disto na §11.

### 9.4 Parâmetros e estatísticas

| Métrica | Valor |
|---|---|
| Partidas | 50 |
| ε | 0.10 |
| MCTS N | 200 |
| Rollout | `heuristic_win` |
| `tactical_root` | True |
| Tempo geração | ~9 min |
| **Total pares** | **856** |
| Pops | 41 (4.8%) |
| Vencedores P1/P2 | 33 / 17 |

Reproduzir: `python generate_dataset.py --games 50`

## 10. ID3 sobre o dataset PopOut + Tree Strategy (`codes/train_tree.py`)

### 10.1 Sweep de `max_depth`

Treinámos ID3 com `max_depth ∈ {3, 5, 8, 10, None}`:

| max_depth | Acc treino | Acc teste | Folhas |
|---|---|---|---|
| 3 | 0.303 | 0.169 | 26 |
| 5 | 0.526 | 0.157 | 156 |
| 8 | 0.876 | 0.221 | 450 |
| **10** | **0.904** | **0.227** | **474** |
| None | 0.904 | 0.227 | 474 |

**Overfitting clássico:** treino sobe a 90%, teste estagna em 22%. Escolhemos `max_depth=10` — sweet-spot empírico, mais profundidade não ajuda.

### 10.2 Tree strategy — fallback em 3 tiers

A árvore aprende correlações estatísticas, **não regras de legalidade**. Pode prever `d3` num estado em que coluna 3 está cheia. O fallback (`decision_tree_builder.py:_fallback_legal_move`):

1. Se a previsão é legal → usa.
2. Senão, escolher entre `legal_moves` o move com **mesma kind** (drop/pop) **mais central**.
3. Senão, drop central qualquer.
4. Última hipótese: primeira jogada legal.

Preserva a "intenção" da árvore tanto quanto possível — se queria drop central e col 3 está cheia, escolhe `d2` ou `d4`.

### 10.3 Tree vs Random — sanity ✅

| Tree | Random | Empates |
|---|---|---|
| **10** | 0 | 0 |

Tempo: tree ~17 µs, random ~10 µs.

### 10.4 Tree vs MCTS-Médio — comparação aluno-professor

| Tree | MCTS-Médio | Empates |
|---|---|---|
| 0 | **6** | 0 |

Tempo: **tree 40 µs vs MCTS 610 ms** = speedup **~15 000×**.

### 10.5 Insight chave

O aluno não supera o professor (esperado em behavioural cloning) mas decide 4 ordens de grandeza mais rápido. **Trade-off força/velocidade** — justifica o uso da árvore em cenários time-bound.

Reproduzir: `python train_tree.py --sweep --vs-random 10 --vs-mcts 6`

In [ ]:
# Demo: 1 partida MCTS-Médio vs Tree
import pickle, random
from game import play_game
from popout import P1
from mcts import mcts_strategy
from decision_tree_builder import tree_strategy

with open('decision_tree.pkl', 'rb') as f:
    tree = pickle.load(f)

final = play_game(
    mcts_strategy(n_simulations=200, rollout='heuristic_win',
                  tactical_root=True, rng=random.Random(0)),
    tree_strategy(tree),
    on_render=lambda _: None, show_intermediate=False, max_turns=200,
)
print(f'Vencedor: P{final.winner}')

## 11. Avaliação experimental (`codes/evaluation.py`)

Esta secção sustenta os **30% técnicos** do enunciado (*"rigour in the performance evaluation"*).

### 11.1 Win-rate matrix — 4 agentes, 2 partidas/célula, alternando lados

Quatro agentes:

| Símbolo | Estratégia | Parâmetros |
|---|---|---|
| `Random` | random uniforme | — |
| `MCTS-E` | MCTS Fácil | N=100, rollout=random |
| `MCTS-M` | MCTS Médio | N=200, rollout=heuristic_win, tactical_root=True |
| `Tree` | ID3 sobre dataset PopOut | max_depth=10, 856 pares |

Resultados (entrada $(i,j)$ = win-rate de $i$ contra $j$):

| A \ B | Random | MCTS-E | MCTS-M | Tree |
|---|---|---|---|---|
| **Random** | 0.50 | 0.00 | 0.00 | 0.00 |
| **MCTS-E** | 1.00 | 0.50 | 0.00 | 0.50 |
| **MCTS-M** | 1.00 | 1.00 | 0.50 | 1.00 |
| **Tree**   | 1.00 | 0.50 | 0.00 | 0.50 |

**Hierarquia:** MCTS-M ≫ MCTS-E ≈ Tree ≫ Random. O ponto crítico é que **Tree e MCTS-Easy têm a mesma força aparente** (cell 0.5/0.5), o que valida a viabilidade do behavioural cloning.

### 11.2 Tempo médio por jogada

| Agente | t/jogada |
|---|---|
| Random | 11 µs |
| **Tree** | **34 µs** |
| MCTS-E | 146 ms |
| MCTS-M | 756 ms |

→ Tree e MCTS-E **mesma força**, mas **Tree é ~4 300× mais rápida**. Este é o argumento principal para defender behavioural cloning como contribuição prática.

### 11.3 Learning curve da árvore

| n_train | Acc teste |
|---|---|
| 137 | 0.211 |
| 274 | 0.187 |
| 411 | 0.222 |
| 548 | 0.216 |

A curva é **plana em ~22%**. Aumentar o dataset não vai resolver — é problema de **representação** (features raw 0/1/2 em cells), não de tamanho.

### 11.4 Outros charts produzidos

`evaluation.py` gera 4 PNGs em `codes/content/`:

- `winrate_matrix.png` — heatmap da matriz acima.
- `tree_learning_curve.png` — curva plana.
- `tree_depth_sensitivity.png` — overfitting clássico (train sobe, test plana).
- `mcts_time_vs_n.png` — tempo do MCTS escala linearmente com N.

Reproduzir: `python evaluation.py [--quick]`

## 12. Conclusões e trabalho futuro

### 12.1 Resultados-chave

1. **MCTS standard** validado numericamente contra o exemplo dos slides (UCB1 = 1.407 / 0.791 / 1.758).
2. **Variações do MCTS** mostram trade-offs claros: `heuristic_win` é o sweet-spot prático; `heuristic_block` é demasiado caro; `max_children=3` poupa 33% sem perder força; C=√2 é o mais rápido.
3. **Tactical lookahead 2-ply** resolveu a observação prática "AI não faz pops mesmo quando popar é a melhor jogada" — apanha vitórias forçadas que rollouts random subestimam.
4. **ID3 em Iris** atinge **94.7% ± 4%** com 5-fold CV e árvore de apenas ~8 folhas, usando `equal_width(k=3)` (sweet-spot do trade-off accuracy/tamanho).
5. **Behavioural cloning** funciona até ao nível MCTS-Easy: Tree iguala MCTS-E em força (0.5/0.5) e é **~4 300× mais rápida** (34 µs vs 146 ms por jogada).
6. **Tree não consegue imitar MCTS-Medium** (perde 0/6) — limite do paradigma com este dataset/features.

### 12.2 Limitações honestas

- **Plafond da árvore PopOut em 22%.** A learning curve é plana — duplicar o dataset não resolve. Triangulação por 3 evidências:
  - Sweep de `max_depth` saturou em 0.22.
  - Learning curve plana entre 137 e 548 amostras.
  - Baseline da classe mais frequente `d3` é 21.3% — estamos só **1 pp acima** do "joga sempre `d3`".
  - **Conclusão:** features raw (cells 0/1/2) não captam padrões úteis. Não é bug, é problema de representação.

- **MCTS-Difícil (N=800) não foi incluído na matriz** por custo computacional (~5 s/jogada × 16 cells = ~80 min wall-clock).

- **ID3 puro sem pruning post-hoc** — usámos `max_depth` como controlo simples em vez de pruning chi-square ou reduced-error.

- **Avaliação com n=2-4 partidas/célula** tem variância alta. Resultados publicáveis exigiriam n≥10.

### 12.3 Trabalho futuro (concreto, por prioridade)

1. **Features derivadas** para o ID3 do PopOut: peças por coluna, comprimento da maior linha consecutiva, ameaças ativas. Hipótese: quebra o plafond de 22% para >50%.
2. **Bitboards** no engine — board em 2 inteiros de 64 bits, detecção de 4-em-linha em ~6 instruções shift+AND. MCTS 5-20× mais rápido.
3. **Pruning chi-square** ou **reduced-error pruning** para o ID3 — controlo de overfitting principled em vez de `max_depth` arbitrário.
4. **MCTS-Difícil na win-rate matrix** — atualmente fora por custo.
5. **Avaliação com n≥10 partidas/célula** para reduzir variância.

A ordem importa: features primeiro, porque a learning curve mostra que mais dados/profundidade não resolvem nada sem mudar a representação.

## 13. Referências

- **Russell, S. & Norvig, P.** — *Artificial Intelligence: A Modern Approach*. Capítulos sobre Adversarial Search (MCTS) e Learning from Observations (Decision Trees, ID3).
- **Allen, J. D. (2010)** — *The Complete Book of Connect-4: History, Strategy, Puzzles*. Sterling. *(Origem das 3 regras especiais do PopOut.)*
- **Browne et al. (2012)** — *A Survey of Monte Carlo Tree Search Methods*. IEEE Transactions on Computational Intelligence and AI in Games.
- **Auer, P., Cesa-Bianchi, N., Fischer, P. (2002)** — *Finite-time Analysis of the Multiarmed Bandit Problem*. Machine Learning 47. *(Base teórica do UCB1.)*
- **Quinlan, J. R. (1986)** — *Induction of Decision Trees*. Machine Learning 1.
- Slides das aulas de IA 2025/2026 — Class 4 (MCTS), Class 7 (Learning).

---

## Reprodutibilidade

Toda a avaliação experimental é reproduzível a partir de `codes/`:

```bash
# Variações do MCTS (Fase 4)
python mcts_variations.py --quick

# Pipeline Iris (Fase 5)
python iris_test.py

# Gerar dataset PopOut (Fase 6, ~10 min)
python generate_dataset.py --games 50

# Treinar tree PopOut (Fase 7)
python train_tree.py --sweep --vs-random 10 --vs-mcts 6

# Avaliação completa (Fase 8)
python evaluation.py [--quick]

# Jogar
python gui.py        # GUI
python game.py       # CLI com menu de modos
```